In [1]:
import csv
import json

def parse_row(row):
    try:
        parsed = dict(row)
        parsed['age'] = int(row['age'])
        parsed['fee_inr'] = int(row['fee_inr'])
        parsed['prior_no_shows'] = int(row['prior_no_shows'])
        parsed['distance_km'] = float(row['distance_km'])
        return parsed
    except ValueError:
        return None

def ingest_appointments(file_path):
    accepted_rows = []
    rejected_rows = []

    with open(file_path, 'r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        headers = reader.fieldnames
        print(f"Headers: {headers}")

        for row in reader:
            parsed_row = parse_row(row)
            if parsed_row is not None:
                accepted_rows.append(parsed_row)
            else:
                rejected_rows.append(row)

    print(f"Total Rows Processed: {len(accepted_rows) + len(rejected_rows)}")
    print(f"Accepted Rows: {len(accepted_rows)} (Target: 988)")
    print(f"Rejected Rows: {len(rejected_rows)} (Target: 32)")

    with open('rejected_rows.json', 'w', encoding='utf-8') as json_file:
        json.dump(rejected_rows, json_file, indent=2)

    return accepted_rows

In [2]:
import pandas as pd

def clean_and_parse_data(file_path):
    df = pd.read_csv(file_path)
    print(f"\nTask 1.2 - Initial shape: {df.shape} (Target: (1020, 14))")
    print("\n--- Info ---")
    df.info()
    print("\n--- Null Counts ---")
    print(df.isnull().sum())
    print(f"\nExact duplicates found: {df.duplicated().sum()} (Target: 20)")

    initial_count = len(df)


    df.drop_duplicates(inplace=True)
    print(f"Task 1.2 (a) - Dropped {initial_count - len(df)} duplicate rows | Remaining: {len(df)} (Target: 1000)")


    df['booking_channel'] = df['booking_channel'].str.strip().str.title()
    print(f"Task 1.2 (b) - Unique channels: {list(df['booking_channel'].unique())} | Remaining: {len(df)}")


    df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
    df['booked_on'] = pd.to_datetime(df['booked_on'])
    print(f"Task 1.2 (c) - Dates converted | Remaining: {len(df)}")


    df.dropna(subset=['appt_date', 'status'], inplace=True)
    print(f"Task 1.2 (d) - Shape after dropping missing appt_date/status: {df.shape} (Target: (990, 14))")

    return df

In [3]:
import numpy as np

def handle_validity_and_outlier(df):
    df = df.copy()


    invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
    print(f"\nTask 1.3 - Invalid ages found: {invalid_age_mask.sum()} (Target: 5)")
    df.loc[invalid_age_mask, 'age'] = np.nan

    missing_age_count = df['age'].isna().sum()
    print(f"Task 1.3 - Total missing ages to impute: {missing_age_count} (Target: 17)")


    df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
    df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())


    for col in ['distance_km', 'fee_inr']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_cap = Q1 - 1.5 * IQR
        upper_cap = Q3 + 1.5 * IQR

        capped_count = ((df[col] < lower_cap) | (df[col] > upper_cap)).sum()
        df[col] = np.clip(df[col], lower_cap, upper_cap)
        print(f"Task 1.3 - {col}: Q1={Q1}, Q3={Q3}, Upper={upper_cap} | Capped={capped_count}")

    df['distance_km'] = df['distance_km'].round(1)
    df['fee_inr'] = df['fee_inr'].round().astype(int)

    return df

In [4]:
import numpy as np
def engineer_features_and_summarize(df):
  df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
  df['no_show'] = np.where(df['status'] == 'No-Show' , 1, 0)
  print(f"\nTask 1.4 - lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()} (Target: 1 to 45)")
  print(f"Task 1.4 - Total no-shows: {df['no_show'].sum()} / {len(df)} ({df['no_show'].mean() * 100:.1f}%) (Target: 218 / 22.0%)")

  X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

  col_means = np.mean(X, axis=0)
  print(f"Task 1.4 (a) - Column means [age, distance_km, fee_inr, lead_days]: {np.round(col_means, 2)}")
  print("               (Targets: [48.17, 9.50, 635.71, 11.82])")

  lead_mask = df['lead_days'].to_numpy()>14
  print(f"Task 1.4 (b) - Rows with lead_days > 14: {lead_mask.sum()} ({lead_mask.mean() * 100:.1f}%) (Target: 296 / 29.9%)")


  dist_arr = df['distance_km'].to_numpy()
  z_score = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
  extreme_z_count = (np.abs(z_score) > 2).sum()
  print(f"Task 1.4 (c) - Rows with |z| > 2 for distance_km: {extreme_z_count} (Target: 53)")

  return df

In [5]:
import requests
import pandas as pd

def fetch_single_clinic_temp(clinic_name , latitude , longitude):
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": latitude,
      "longitude": longitude,
      "hourly": "temperature_2m",
      "forecast_days": 1
  }
  failure_reason = "Unknown Error"


  for attempt in range(2):
    try:
      response = requests.get(url, params=params, timeout = 10)
      if response.status_code == 200:
        data = response.json()
        hourly = data.get("hourly")
        if hourly and "temperature_2m" in hourly and len(hourly["temperature_2m"]) > 0:
          return hourly["temperature_2m"][0]
        failure_reason = "Missing 'hourly' or 'temperature_2m' in API response"
      else:
        failure_reason = f"HTTP status {response.status_code}"
    except requests.exception.RequestException as err:
      failure_reason = str(err)

  print(f"Task 1.5 - Warning: Could not fetch temperature for '{clinic_name}' ({failure_reason})")
  return None

def enrich_clinics(clinics_path, output_csv="clinics_enriched.csv"):
    clinics_df = pd.read_csv(clinics_path)
    temps = []

    for _, row in clinics_df.iterrows():
        temp = fetch_single_clinic_temp(row['clinic_name'], row['latitude'], row['longitude'])
        temps.append(temp)

    clinics_df['temp_c'] = temps
    clinics_df.to_csv(output_csv, index=False)
    print(f"\nTask 1.5 - Saved {output_csv} ({len(clinics_df)} rows):")
    print(clinics_df[['clinic_id', 'clinic_name', 'city', 'temp_c']])

    return clinics_df


In [6]:
def export_clean_sql(df, clinics_df, clean_csv="appointments_clean.csv" , sql_path="load_data.sql"):
  df_out = df.sort_values('appointment_id').copy()
  df_out['booked_on']=df_out['booked_on'].dt.strftime('%Y-%m-%d')
  df_out['appt_date']=df_out['appt_date'].dt.strftime('%Y-%m-%d')

  ordered_cols = [
        'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
        'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
        'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
        'lead_days', 'no_show'
    ]

  df_out = df_out[ordered_cols]
  unique_appts = df_out['appointment_id'].nunique()
  total_missing = df_out.isnull().sum().sum()

  print(f"\nTask 1.6 - Unique appointment_id count: {unique_appts} (Target: 990)")
  print(f"Task 1.6 - Missing values across 16 columns: {total_missing} (Target: 0)")

  assert unique_appts == len(df_out),"Error: appointment_id is not unique!"
  assert total_missing == 0 ,"Error: Found missing values in export DataFrame!"

  df_out.to_csv(clean_csv, index=False)
  print(f"Task 1.6 - Exported {clean_csv} ({df_out.shape})")

  sql_statements = []

  for _, r in clinics_df.iterrows():
        temp_sql = "NULL" if pd.isna(r['temp_c']) else f"{r['temp_c']}"
        stmt = (
            f"INSERT INTO clinics VALUES ("
            f"'{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',"
            f"{r['latitude']},{r['longitude']},{temp_sql});"
        )
        sql_statements.append(stmt)

  for _ , r in df_out.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
        f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',"
        f"{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}',"
        f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},"
        f"'{r['status']}',{r['lead_days']},{r['no_show']});"
    )
    sql_statements.append(stmt)

  with open(sql_path, 'w' , encoding = 'utf-8') as file:
    file.write('\n'.join(sql_statements) + "\n")

  print(f"Task 1.6 - Wrote {len(sql_statements)} INSERT statements to {sql_path} (Target: 998)")
  return df_out

In [7]:
accepted_rows = ingest_appointments("/content/appointments.csv")
df_step1 = clean_and_parse_data("/content/appointments.csv")
df_step2 = handle_validity_and_outlier(df_step1)
df_step3 = engineer_features_and_summarize(df_step2)
clinics_df = enrich_clinics("/content/clinics.csv")
df_clean = export_clean_sql(df_step3, clinics_df)

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows Processed: 1020
Accepted Rows: 988 (Target: 988)
Rejected Rows: 32 (Target: 32)

Task 1.2 - Initial shape: (1020, 14) (Target: (1020, 14))

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel 

In [8]:
#Part 2 — SQL: verify the numbers and find where no-shows cluster (22 marks)


In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("SQLite.db")
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript("""
  DROP TABLE IF EXISTS appointments;
  DROP TABLE IF EXISTS clinics;

  CREATE TABLE clinics (
  clinic_id VARCHAR(5) PRIMARY KEY,
  clinic_name VARCHAR(40) NOT NULL,
  city VARCHAR(20) NOT NULL,
  tier VARCHAR(10),
  latitude DECIMAL(8,4),
  longitude DECIMAL(8,4),
  temp_c DECIMAL(5,1)
  );

  CREATE TABLE appointments (
  appointment_id VARCHAR(8) PRIMARY KEY,
  patient_id VARCHAR(8) NOT NULL,
  clinic_id VARCHAR(5) NOT NULL,
  booked_on DATE,
  appt_date DATE,
  age INT,
  gender VARCHAR(1),
  distance_km DECIMAL(5,1),
  fee_inr INT,
  booking_channel VARCHAR(10),
  visit_type VARCHAR(12),
  reminder_sent VARCHAR(3),
  prior_no_shows INT,
  status VARCHAR(10),
  lead_days INT,
  no_show INT,
  FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id)
  );


""")

with open("load_data.sql" , "r", encoding = "utf-8") as f:
  conn.executescript(f.read())


print(pd.read_sql_query("SELECT COUNT(*) AS clinics FROM clinics;", conn))
print(pd.read_sql_query("SELECT COUNT(*) AS appointments, SUM(no_show) AS no_shows FROM appointments;", conn))

try:
    conn.execute("INSERT INTO appointments VALUES ('A0001','P999','C01','2025-01-01','2025-01-05',30,'M',5.0,500,'App','Consultation','Yes',0,'Attended',4,0);")
except sqlite3.IntegrityError as e:
    print("Duplicate key error:", e)

   clinics
0        8
   appointments  no_shows
0           990       218
Duplicate key error: UNIQUE constraint failed: appointments.appointment_id


In [11]:
import sqlite3
import pandas as pd

def run_part_2_queries(conn):
    # ==========================================
    # Task 2.2 — No-show rate by clinic
    # ==========================================
    q_2_2_having = """
    SELECT
        c.clinic_name,
        c.city,
        COUNT(*) AS appointments,
        SUM(a.no_show) AS no_shows,
        ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
    FROM appointments a
    JOIN clinics c ON a.clinic_id = c.clinic_id
    GROUP BY c.clinic_id, c.clinic_name, c.city
    HAVING COUNT(*) >= 95
    ORDER BY no_show_pct DESC;
    """
    print("--- Task 2.2 (Clinics with >= 95 appointments) ---")
    print(pd.read_sql_query(q_2_2_having, conn).to_string(index=False))

    q_2_2_all = """
    SELECT
        c.clinic_name,
        c.city,
        COUNT(*) AS appointments,
        SUM(a.no_show) AS no_shows,
        ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
    FROM appointments a
    JOIN clinics c ON a.clinic_id = c.clinic_id
    GROUP BY c.clinic_id, c.clinic_name, c.city
    ORDER BY no_show_pct DESC;
    """
    print("\n--- Task 2.2 (All 8 clinics, no HAVING) ---")
    sql_all_df = pd.read_sql_query(q_2_2_all, conn)
    print(sql_all_df.to_string(index=False))

    # Python groupby verification
    appts_df = pd.read_csv("appointments_clean.csv")
    clinics_df = pd.read_csv("clinics_enriched.csv")
    merged = appts_df.merge(clinics_df, on="clinic_id")
    py_grouped = (
        merged.groupby(["clinic_name", "city"], as_index=False)
        .agg(appointments=("no_show", "count"), no_shows=("no_show", "sum"))
    )
    py_grouped["no_show_pct"] = (100.0 * py_grouped["no_shows"] / py_grouped["appointments"]).round(1)
    py_grouped = py_grouped.sort_values("no_show_pct", ascending=False).reset_index(drop=True)
    print("\n--- Task 2.2 Python Groupby Verification ---")
    print(py_grouped.to_string(index=False))

    # ==========================================
    # Task 2.3 — Does booking far ahead matter?
    # ==========================================
    q_2_3 = """
    SELECT
        CASE
            WHEN lead_days BETWEEN 1 AND 7 THEN '1-7'
            WHEN lead_days BETWEEN 8 AND 14 THEN '8-14'
            WHEN lead_days BETWEEN 15 AND 30 THEN '15-30'
            ELSE '31+'
        END AS lead_band,
        COUNT(*) AS appointments,
        SUM(no_show) AS no_shows,
        ROUND(100.0 * SUM(no_show) / COUNT(*), 1) AS no_show_pct
    FROM appointments
    GROUP BY lead_band
    ORDER BY MIN(lead_days);
    """
    print("\n--- Task 2.3 (No-show rate by lead_days band) ---")
    print(pd.read_sql_query(q_2_3, conn).to_string(index=False))

    # ==========================================
    # Task 2.4 — Month-by-month trend (CTE + LAG)
    # ==========================================
    q_2_4 = """
    WITH monthly AS (
        SELECT
            SUBSTR(appt_date, 1, 7) AS month,
            COUNT(*) AS appointments,
            ROUND(100.0 * SUM(no_show) / COUNT(*), 1) AS no_show_pct
        FROM appointments
        GROUP BY SUBSTR(appt_date, 1, 7)
    )
    SELECT
        month,
        appointments,
        no_show_pct,
        ROUND(no_show_pct - LAG(no_show_pct) OVER (ORDER BY month), 1) AS pct_point_change
    FROM monthly
    ORDER BY month;
    """
    print("\n--- Task 2.4 (Month-by-month trend) ---")
    print(pd.read_sql_query(q_2_4, conn).to_string(index=False))

    # ==========================================
    # Task 2.5 — Rank clinics inside each city
    # ==========================================
    q_2_5 = """
    WITH clinic_rates AS (
        SELECT
            c.clinic_name AS clinic,
            c.city,
            ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
        FROM appointments a
        JOIN clinics c ON a.clinic_id = c.clinic_id
        GROUP BY c.clinic_id, c.clinic_name, c.city
    )
    SELECT
        clinic,
        city,
        no_show_pct,
        RANK() OVER (PARTITION BY city ORDER BY no_show_pct DESC) AS rank_in_city,
        CASE
            WHEN no_show_pct > (SELECT 100.0 * SUM(no_show) / COUNT(*) FROM appointments) THEN 'above'
            WHEN no_show_pct < (SELECT 100.0 * SUM(no_show) / COUNT(*) FROM appointments) THEN 'below'
            ELSE 'equal'
        END AS vs_overall
    FROM clinic_rates
    ORDER BY city, rank_in_city;
    """
    print("\n--- Task 2.5 (Clinic rank inside each city vs overall 22.02%) ---")
    print(pd.read_sql_query(q_2_5, conn).to_string(index=False))

run_part_2_queries(conn)

--- Task 2.2 (Clinics with >= 95 appointments) ---
             clinic_name   city  appointments  no_shows  no_show_pct
    Hadapsar Care Centre   Pune           143        39         27.3
    Malviya Nagar Clinic Jaipur           142        38         26.8
   Kothrud Family Clinic   Pune           160        36         22.5
      Vijay Nagar Clinic Indore           101        21         20.8
Edappally Medical Centre  Kochi           142        23         16.2
   Vaishali Nagar Clinic Jaipur           130        18         13.8

--- Task 2.2 (All 8 clinics, no HAVING) ---
             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
E

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def run_part_3_ml(clean_csv="appointments_clean.csv"):
    df = pd.read_csv(clean_csv)
    appt_ids = df["appointment_id"].copy()

    # ==========================================
    # Task 3.1 — Features, split and baseline
    # ==========================================
    feature_cols = [
        "age", "distance_km", "fee_inr", "lead_days",
        "prior_no_shows", "reminder_sent", "booking_channel", "visit_type"
    ]
    X = df[feature_cols].copy()
    X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
    X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
    y = df["no_show"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    baseline_acc = (y_test == 0).mean()
    print(f"Task 3.1 - X shape: {X.shape} (Target cols: 11)")
    print(f"Task 3.1 - Train rows: {len(X_train)}, Test rows: {len(X_test)}")
    print(f"Task 3.1 - Test no-shows: {y_test.sum()} (Target: 49)")
    print(f"Task 3.1 - Baseline accuracy (always Attended): {baseline_acc:.4f} (Target: 0.7525)")

    # ==========================================
    # Task 3.2 — Logistic regression
    # ==========================================
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train_s, y_train)
    lr_preds = lr.predict(X_test_s)
    lr_probs = lr.predict_proba(X_test_s)[:, 1]

    lr_acc = accuracy_score(y_test, lr_preds)
    lr_prec = precision_score(y_test, lr_preds)
    lr_rec = recall_score(y_test, lr_preds)
    lr_f1 = f1_score(y_test, lr_preds)
    lr_auc = roc_auc_score(y_test, lr_probs)

    print("\n--- Task 3.2: Logistic Regression (Test Set) ---")
    print(f"Accuracy: {lr_acc:.4f} | Precision: {lr_prec:.4f} | Recall: {lr_rec:.4f} | F1: {lr_f1:.4f} | ROC-AUC: {lr_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, lr_preds))

    coef_df = pd.DataFrame({
        "feature": X.columns,
        "coef": lr.coef_[0],
        "scale": scaler.scale_
    })
    print("\nCoefficients:\n", coef_df[["feature", "coef"]].to_string(index=False))

    rem_row = coef_df[coef_df["feature"] == "reminder_sent"].iloc[0]
    odds_scaled = np.exp(rem_row["coef"])
    odds_unscaled = np.exp(rem_row["coef"] / rem_row["scale"])
    print(f"\nreminder_sent -> exp(coef) [1 SD]: {odds_scaled:.4f} (Target: ~0.56)")
    print(f"reminder_sent -> exp(coef / scale) [No->Yes]: {odds_unscaled:.4f} (Target: ~0.31, ~69% lower odds)")

    # ==========================================
    # Task 3.3 — Decision tree and overfitting
    # ==========================================
    dt_full = DecisionTreeClassifier(random_state=42)
    dt_full.fit(X_train, y_train)
    print("\n--- Task 3.3: Decision Trees ---")
    print(f"Full Tree (depth={dt_full.get_depth()}) -> Train Acc: {dt_full.score(X_train, y_train):.4f}, Test Acc: {dt_full.score(X_test, y_test):.4f}")

    dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_3.fit(X_train, y_train)
    dt3_preds = dt_3.predict(X_test)
    dt3_probs = dt_3.predict_proba(X_test)[:, 1]
    dt3_acc = accuracy_score(y_test, dt3_preds)
    dt3_rec = recall_score(y_test, dt3_preds)
    dt3_auc = roc_auc_score(y_test, dt3_probs)
    print(f"Depth-3 Tree -> Train Acc: {dt_3.score(X_train, y_train):.4f}, Test Acc: {dt3_acc:.4f}")

    # ==========================================
    # Task 3.4 — Random forest & comparison table
    # ==========================================
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    rf_probs = rf.predict_proba(X_test)[:, 1]
    rf_acc = accuracy_score(y_test, rf_preds)
    rf_rec = recall_score(y_test, rf_preds)
    rf_auc = roc_auc_score(y_test, rf_probs)

    imp_df = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
    print("\n--- Task 3.4: Random Forest & Model Comparison ---")
    print("Top 3 RF Feature Importances:\n", imp_df.head(3).to_string(index=False))

    comp_df = pd.DataFrame([
        {"Model": "Logistic Regression", "Accuracy": round(lr_acc, 4), "Recall": round(lr_rec, 4), "ROC-AUC": round(lr_auc, 4)},
        {"Model": "Decision Tree (depth=3)", "Accuracy": round(dt3_acc, 4), "Recall": round(dt3_rec, 4), "ROC-AUC": round(dt3_auc, 4)},
        {"Model": "Random Forest (100)", "Accuracy": round(rf_acc, 4), "Recall": round(rf_rec, 4), "ROC-AUC": round(rf_auc, 4)},
    ])
    print("\nTest Comparison Table (For Reporting Only):\n", comp_df.to_string(index=False))

    # ==========================================
    # Task 3.5 — Tune, choose model & threshold, test once
    # ==========================================
    grid = GridSearchCV(
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
        cv=5,
        scoring="roc_auc"
    )
    grid.fit(X_train, y_train)
    print("\n--- Task 3.5: CV Model Selection & Threshold Tuning ---")
    print(f"(a) Tuned RF best_params_: {grid.best_params_} | best_score_ (CV AUC): {grid.best_score_:.4f}")

    lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
    cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
    cv_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
    cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
    print(f"(b) 5-Fold CV ROC-AUC -> LR: {cv_lr:.4f} | DT3: {cv_dt3:.4f} | RF: {cv_rf:.4f} | Tuned RF: {grid.best_score_:.4f}")

    # Out-of-fold probabilities on training set for winner (Logistic Regression)
    oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
    print("\n(c) Out-of-Fold Threshold Comparison (Training Set):")
    for thresh in [0.5, 0.35]:
        oof_flag = (oof_probs >= thresh).astype(int)
        p = precision_score(y_train, oof_flag)
        r = recall_score(y_train, oof_flag)
        f1 = f1_score(y_train, oof_flag)
        print(f"    Threshold {thresh:.2f} -> Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}, Flagged: {oof_flag.sum()}")

    # Final single evaluation on test set at chosen threshold 0.35
    chosen_thresh = 0.35
    lr_pipe.fit(X_train, y_train)
    test_probs = lr_pipe.predict_proba(X_test)[:, 1]
    test_flags = (test_probs >= chosen_thresh).astype(int)

    print("\n(d) Final Test Evaluation (Logistic Regression @ threshold 0.35):")
    print(f"    Precision: {precision_score(y_test, test_flags):.4f}")
    print(f"    Recall:    {recall_score(y_test, test_flags):.4f}")
    print(f"    Flagged:   {test_flags.sum()} / {len(y_test)}")
    print(f"    ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")

    # (e) Save test_risk_scores.csv
    scores_df = pd.DataFrame({
        "appointment_id": appt_ids.loc[X_test.index].values,
        "no_show": y_test.values,
        "risk_prob": np.round(test_probs, 4),
        "flag": test_flags
    })
    scores_df.to_csv("test_risk_scores.csv", index=False)
    print(f"\n(e) Saved test_risk_scores.csv ({len(scores_df)} rows, {scores_df['flag'].sum()} flagged).")
    return scores_df

scores_df = run_part_3_ml()

Task 3.1 - X shape: (990, 11) (Target cols: 11)
Task 3.1 - Train rows: 792, Test rows: 198
Task 3.1 - Test no-shows: 49 (Target: 49)
Task 3.1 - Baseline accuracy (always Attended): 0.7525 (Target: 0.7525)

--- Task 3.2: Logistic Regression (Test Set) ---
Accuracy: 0.7879 | Precision: 0.6296 | Recall: 0.3469 | F1: 0.4474 | ROC-AUC: 0.7889
Confusion Matrix:
 [[139  10]
 [ 32  17]]

Coefficients:
                 feature      coef
                    age -0.253880
            distance_km  0.253448
                fee_inr  0.452761
              lead_days  0.614471
         prior_no_shows  0.649612
          reminder_sent -0.580669
    booking_channel_App -0.046804
  booking_channel_Phone  0.215881
booking_channel_Walk-In -0.186364
visit_type_Consultation  0.015658
   visit_type_Follow-up -0.015658

reminder_sent -> exp(coef) [1 SD]: 0.5595 (Target: ~0.56)
reminder_sent -> exp(coef / scale) [No->Yes]: 0.3094 (Target: ~0.31, ~69% lower odds)

--- Task 3.3: Decision Trees ---
Full Tree (dept

In [ ]:
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are an outpatient clinic operations analyst at CareLoop Clinics in India. "
    "Extract structured insights from post-appointment call-centre notes accurately and objectively."
)

def build_craft_prompt(note_text):
    return (
        "Context: A patient missed a scheduled outpatient appointment at CareLoop Clinics, and a call-centre agent logged a brief follow-up note.\n"
        "Role: Healthcare operations analyst.\n"
        "Action: Classify the primary reason for the missed visit into exactly one of these allowed root_cause labels: "
        "forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. "
        "If the note is ambiguous, unclear, or does not fit the first five categories, choose 'other' rather than guessing.\n"
        "Tone: Concise, neutral, and operational.\n"
        "Format: Return valid JSON with root_cause, confidence (0.0 to 1.0), summary (max 25 words), and recommended_action.\n\n"
        f"Call-centre note: \"{note_text}\""
    )

api_call_count = 0

def call_gemini_raw(prompt_text):
    global api_call_count
    api_call_count += 1
    config_kwargs = dict(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
    )
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(
                **config_kwargs,
                thinking_config=types.ThinkingConfig(thinking_level="minimal")
            ),
        )
    except Exception:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(**config_kwargs),
        )
    return response.text

# Task 4.1
def analyse_note(note_text):
    raw_text = call_gemini_raw(build_craft_prompt(note_text))
    return json.loads(raw_text)

notes_df = pd.read_csv("followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_result = analyse_note(n001_text)
print("Task 4.1 (N001):", json.dumps(n001_result, indent=2))

# Task 4.2
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def process_first_10_notes(notes_df, out_csv="notes_analysed.csv"):
    results = []
    subset = notes_df.head(10)

    for idx, row in subset.iterrows():
        note_id = row["note_id"]
        appt_id = row["appointment_id"]
        note_text = row["note_text"]

        if pd.isna(note_text) or str(note_text).strip() == "":
            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "No note recorded",
                "recommended_action": "Re-call patient",
                "validation_status": "skipped_blank",
                "raw_response": ""
            })
            continue

        raw_reply = ""
        try:
            raw_reply = call_gemini_raw(build_craft_prompt(str(note_text)))
            parsed = json.loads(raw_reply)
            insight = FollowupInsight(**parsed)
            is_allowed = insight.root_cause in ALLOWED_CAUSES
            status = "valid" if is_allowed else "invalid"

            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": parsed.get("root_cause", ""),
                "confidence": parsed.get("confidence", ""),
                "summary": parsed.get("summary", ""),
                "recommended_action": parsed.get("recommended_action", ""),
                "validation_status": status,
                "raw_response": raw_reply
            })
        except Exception:
            parsed_partial = {}
            try:
                parsed_partial = json.loads(raw_reply) if raw_reply else {}
            except Exception:
                pass
            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": parsed_partial.get("root_cause", ""),
                "confidence": parsed_partial.get("confidence", ""),
                "summary": parsed_partial.get("summary", ""),
                "recommended_action": parsed_partial.get("recommended_action", ""),
                "validation_status": "invalid",
                "raw_response": raw_reply
            })

        time.sleep(13)

    analysed_df = pd.DataFrame(results)
    analysed_df.to_csv(out_csv, index=False)
    print(f"\nTask 4.2 - Saved {out_csv} ({len(analysed_df)} rows):")
    print(analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
    return analysed_df

analysed_df = process_first_10_notes(notes_df)

# Task 4.3
print("\n--- Task 4.3 (a): Deliberate Pydantic Validation Errors ---")
bad_cases = [
    {"root_cause": "other", "confidence": 1.7, "summary": "Patient missed visit", "recommended_action": "Call back"},
    {"root_cause": "other", "confidence": 0.8, "summary": "Patient missed visit"}
]
for i, bad in enumerate(bad_cases, 1):
    try:
        FollowupInsight(**bad)
    except ValidationError as e:
        print(f"Bad case {i} caught:\n{e}\n")

print("--- Task 4.3 (b): V1 vs V2 Prompt Comparison on N007 & N009 ---")
comparison_rows = []
for nid in ["N007", "N009"]:
    ntext = notes_df.loc[notes_df["note_id"] == nid, "note_text"].iloc[0]

    raw_v1 = call_gemini_raw(f"Classify this note: {ntext}")
    time.sleep(13)
    d_v1 = json.loads(raw_v1)
    rc_v1 = d_v1.get("root_cause", "")
    comparison_rows.append({
        "note_id": nid, "prompt": "V1 (Minimal)",
        "root_cause": rc_v1, "in_allowed_list": rc_v1 in ALLOWED_CAUSES
    })

    raw_v2 = call_gemini_raw(build_craft_prompt(ntext))
    time.sleep(13)
    d_v2 = json.loads(raw_v2)
    rc_v2 = d_v2.get("root_cause", "")
    comparison_rows.append({
        "note_id": nid, "prompt": "V2 (CRAFT)",
        "root_cause": rc_v2, "in_allowed_list": rc_v2 in ALLOWED_CAUSES
    })

print(pd.DataFrame(comparison_rows).to_string(index=False))
print(f"\nTask 4.3 (c) - Total Gemini API calls used across Tasks 4.1–4.3: {api_call_count}")

# Task 4.4
clean_appts = pd.read_csv("appointments_clean.csv")
merged_notes = analysed_df.merge(clean_appts, on="appointment_id", how="inner")
ct = pd.crosstab(merged_notes["root_cause"], merged_notes["reminder_sent"])
print("\n--- Task 4.4 (a): Cross-tab of root_cause vs reminder_sent ---")
print(ct)